In [7]:
using Pkg
# Pkg.add(["Flux", "MLDatasets"])

using Random
using Statistics
using MLDatasets
using Flux
using Flux: logitcrossentropy, onehotbatch, onecold, DataLoader

In [8]:
const lr = 0.01f0
const num_epochs = 3
const batch_size = 1
const dropout_p = 0.4f0
const rand_seed = 1

Random.seed!(rand_seed)

TaskLocalRNG()

### Data loading

In [9]:
println("[x] Loading FashionMNIST data...")
train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

x_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
x_test  = Float32.(reshape(test_data.features,  28, 28, 1, :))

y_train = onehotbatch(train_data.targets, 0:9)
y_test  = onehotbatch(test_data.targets, 0:9)

train_loader = DataLoader((x_train, y_train), batchsize=batch_size, shuffle=true)

[x] Loading FashionMNIST data...


60000-element DataLoader(::Tuple{Array{Float32, 4}, OneHotArrays.OneHotMatrix{UInt32, Vector{UInt32}}}, shuffle=true)
  with first element:
  (28×28×1×1 Array{Float32, 4}, 10×1 OneHotMatrix(::Vector{UInt32}) with eltype Bool,)

### Model and optimizer

In [ ]:
model = Chain(
    Conv((3, 3), 1 => 6, pad=1),
    MaxPool((2, 2)),
    Conv((3, 3), 6 => 16, pad=1),
    MaxPool((2, 2)),
    Flux.flatten,
    Dense(784 => 84, relu),
    Dropout(dropout_p),
    Dense(84 => 10)
)

# SGD
opt = Flux.setup(Descent(lr), model)

function test_accuracy(model, x, y)
    Flux.testmode!(model)  # Wyłącza Dropout
    preds = model(x)
    Flux.trainmode!(model) # Włącza Dropout z powrotem
    return mean(onecold(preds) .== onecold(y))
end

test_accuracy (generic function with 1 method)

### Training loop and test

In [11]:
println("[x] Accuracy before training: ", round(test_accuracy(model, x_test, y_test) * 100, digits=2), "%")
println("\n[x] Training started...")

for epoch in 1:num_epochs
    total_loss = 0.0f0
    
    t = @elapsed begin
        for (x_batch, y_batch) in train_loader
            loss, grads = Flux.withgradient(model) do m
                preds = m(x_batch)
                logitcrossentropy(preds, y_batch)
            end
            
            # Aktualizacja wag
            Flux.update!(opt, model, grads[1])
            total_loss += loss
        end
    end
    
    println("[+] Epoch $epoch | Loss: $(round(total_loss, digits=3)) | Time: $(round(t, digits=2))s")
end

println("\n[x] Accuracy after training: ", round(test_accuracy(model, x_test, y_test) * 100, digits=2), "%")

[x] Accuracy before training: 5.97%

[x] Training started...
[+] Epoch 1 | Loss: 30687.785 | Time: 40.2s
[+] Epoch 2 | Loss: 23837.191 | Time: 37.04s
[+] Epoch 3 | Loss: 22426.742 | Time: 36.81s

[x] Accuracy after training: 86.15%
